In [0]:
import dlt
from pyspark.sql.functions import col, window, max, avg, round

In [0]:
@dlt.table(
    name="03_gold.buoy_stats", 
    comment="24-hour avg and max water_temperature, salinity, current_speed, wind_speed, wave_height and wave_period with window start/end"
)
def buoy_stats():
    # Apply a 2-minute watermark for late data
    wtemp = (
        dlt.read_stream("02_silver.water_temperature")
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withWatermark("ingestion_timestamp", "2 minutes")
    )
    sal = (
        dlt.read_stream("02_silver.salinity")
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withWatermark("ingestion_timestamp", "2 minutes")
    )
    cspeed = (
        dlt.read_stream("02_silver.current_speed")
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withWatermark("ingestion_timestamp", "2 minutes")
    )
    wspeed = (
        dlt.read_stream("02_silver.wind_speed")
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withWatermark("ingestion_timestamp", "2 minutes")
    )
    wheight = (
        dlt.read_stream("02_silver.wave_height")
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withWatermark("ingestion_timestamp", "2 minutes")
    )
    wperiod = (
        dlt.read_stream("02_silver.wave_period")
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withWatermark("ingestion_timestamp", "2 minutes")
    )

    # Compute 24-hour windowed aggregations (24 hours of simulated time = 24 minutes of actual time)
    wtemp_agg = (
        wtemp
        .groupBy(
            window(col("ingestion_timestamp"), "24 minutes"),
            col("buoy_id")
        )
        .agg(
            avg(col("water_temperature")).alias("avg_water_temperature"),
            max(col("water_temperature")).alias("max_water_temperature")
        )
        .select(
            col("buoy_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_water_temperature"),
            col("max_water_temperature")
        )
    )

    sal_agg = (
        sal
        .groupBy(
            window(col("ingestion_timestamp"), "24 minutes"),
            col("buoy_id")
        )
        .agg(
            avg(col("salinity")).alias("avg_salinity"),
            max(col("salinity")).alias("max_salinity")
        )
        .select(
            col("buoy_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_salinity"),
            col("max_salinity")
        )
    )

    cspeed_agg = (
        cspeed
        .groupBy(
            window(col("ingestion_timestamp"), "24 minutes"),
            col("buoy_id")
        )
        .agg(
            avg(col("current_speed")).alias("avg_current_speed"),
            max(col("current_speed")).alias("max_current_speed")
        )
        .select(
            col("buoy_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_current_speed"),
            col("max_current_speed")
        )
    )

    wspeed_agg = (
        wspeed
        .groupBy(
            window(col("ingestion_timestamp"), "24 minutes"),
            col("buoy_id")
        )
        .agg(
            avg(col("wind_speed")).alias("avg_wind_speed"),
            max(col("wind_speed")).alias("max_wind_speed")
        )
        .select(
            col("buoy_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_wind_speed"),
            col("max_wind_speed")
        )
    )

    wheight_agg = (
        wheight
        .groupBy(
            window(col("ingestion_timestamp"), "24 minutes"),
            col("buoy_id")
        )
        .agg(
            avg(col("wave_height")).alias("avg_wave_height"),
            max(col("wave_height")).alias("max_wave_height")
        )
        .select(
            col("buoy_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_wave_height"),
            col("max_wave_height")
        )
    )

    wperiod_agg = (
        wperiod
        .groupBy(
            window(col("ingestion_timestamp"), "24 minutes"),
            col("buoy_id")
        )
        .agg(
            avg(col("wave_period")).alias("avg_wave_period"),
            max(col("wave_period")).alias("max_wave_period")
        )
        .select(
            col("buoy_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_wave_period"),
            col("max_wave_period")
        )
    )

    return (
        wtemp_agg.alias("wt")
        .join(
            sal_agg.alias("s"),
            on=["buoy_id", "window_start", "window_end"],
            how="inner"
        )
        .join(
            cspeed_agg.alias("cs"),
            on=["buoy_id", "window_start", "window_end"],
            how="inner"
        )
        .join(
            wspeed_agg.alias("ws"),
            on=["buoy_id", "window_start", "window_end"],
            how="inner"
        )
        .join(
            wheight_agg.alias("wh"),
            on=["buoy_id", "window_start", "window_end"],
            how="inner"
        )
        .join(
            wperiod_agg.alias("wp"),
            on=["buoy_id", "window_start", "window_end"],
            how="inner"
        )
        .select(
            col("buoy_id"),
            col("window_start"),
            col("window_end"),
            round(col("avg_water_temperature"), 2).alias("avg_water_temperature"),
            col("max_water_temperature"),
            round(col("avg_salinity"), 2).alias("avg_salinity"),
            col("max_salinity"),
            round(col("avg_current_speed"), 2).alias("avg_current_speed"),
            col("max_current_speed"),
            round(col("avg_wind_speed"), 2).alias("avg_wind_speed"),
            col("max_wind_speed"),
            round(col("avg_wave_height"), 2).alias("avg_wave_height"),
            col("max_wave_height"),
            round(col("avg_wave_period"), 2).alias("avg_wave_period"),
            col("max_wave_period")
        )
    )
    

